In [ ]:
# -*- coding: utf-8 -*-
import os, sys, json, glob, pickle

import torch
import torch.nn.functional as F
from types import SimpleNamespace
from hydra.utils import instantiate
from tqdm import tqdm
import pdb
import numpy as np

# --- Path setup (relative to notebook location) ---
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
TOKENIZER_DIR = os.path.dirname(NOTEBOOK_DIR)  # 00_Hierarchical_Tokenizer
REPO_DIR = os.path.dirname(TOKENIZER_DIR)  # TimeTok_Repo

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, TOKENIZER_DIR)

from utils.misc import detect_bf16_support, get_bf16_context
from data.UCR_data_factory import ucr_data_provider
from data.Nasdaq_factory import nasdaq_data_provider
from data.ETTh1_factory import etth1_data_provider
from TimeTok_wrapper import TimeTok

# This is simply used as identifier
data_dict = {
    'ECG5000': 0,
    'ItalyPowerDemand': 1,
    'Nasdaq': 2,
    'ETTh1': 3
}

Pz        = 3  # patch size (1D)
TARGET_DATASET = "ECG5000"
TOKENIZER_DATASET = "ECG5000" # If you want to use UTSD tokenizer, set as UTSD. 

return_eval_keep_k = False ### IMPORTANT Parameter. True for foundation model, False for individual model

dataset_id = data_dict[TARGET_DATASET] 

# Paths (relative to repository structure)
DATA_DIR = os.path.join(REPO_DIR, "data")
UCR_ROOT = os.path.join(DATA_DIR, "UCRConverted")

CKPT_DIR = os.path.join(REPO_DIR, "03_Shared", "tokenizer_weights", TOKENIZER_DATASET)

OUT_DIR   = os.path.join(REPO_DIR, "03_Shared", "tokenized_sequences", f"{TOKENIZER_DATASET}_tokenizer", f"{TARGET_DATASET}_pz{Pz}")

DEVICE    = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ENABLE_BF16 = detect_bf16_support()

In [ ]:
def pad_to_multiple_L(x, multiple: int):
    """x: [B,C,L] -> reflect pad"""
    L = x.shape[-1]
    need = (multiple - (L % multiple)) % multiple
    if need == 0: return x, 0
    return F.pad(x, (0, need), mode="reflect"), need

def tokenize_split(data_loader, split_name, tsgen, out_path, dataset_name, patch_size, enable_bf16, dataset_id, return_eval_keep_k=True):
    """save to json"""
    if os.path.exists(out_path):
        os.remove(out_path)
    dataset_id = torch.tensor(dataset_id, dtype=torch.long)
    sample_id = 0
    with torch.no_grad(), get_bf16_context(enable_bf16):
        with open(out_path, "a") as fout:
            for batch in tqdm(data_loader, desc=f"Tokenizing {dataset_name}/{split_name}"):
                # Batch Unpack
                if isinstance(batch, (list, tuple)):
                    x, y = batch[0], batch[1]
                else:
                    x, y = batch, None

                x = x.to(DEVICE)  # [B,C,L]
                x_pad, _ = pad_to_multiple_L(x, patch_size)

                # List[Tensor] of shape [1, L_tokens]
                if return_eval_keep_k:
                    tokens_list, eval_keep_k = tsgen.tokenize(x_pad, return_eval_keep_k=return_eval_keep_k)
                else:
                    tokens_list = tsgen.tokenize(x_pad, return_eval_keep_k=return_eval_keep_k)
                    eval_keep_k = [-1] * len(tokens_list)

                B = len(tokens_list)
                for i in range(B):
                    tokens_1xL = tokens_list[i]                 # [1, L_tokens]
                    tokens = tokens_1xL.squeeze(0).to("cpu")    # [L_tokens]
                    tokens_int = tokens.long().tolist()         # List[int]
                    eval_keep_k_sample = eval_keep_k[i]

                    label_val = None
                    if y is not None:
                        if dataset_name in ["Nasdaq", "ETTh1"]:
                            values = list(np.array(y[i].squeeze()))
                            label_val = [float(x) for x in values]
                        else:
                            label_val = int(y[i].item()) if torch.is_tensor(y) else int(y[i])

                    rec = {
                        "dataset": dataset_name,
                        "split": split_name,
                        "eval_keep_k": eval_keep_k_sample,
                        "sample_id": sample_id,
                        "dataset_id": dataset_id.item(),
                        "label": label_val,
                        "token_len": len(tokens_int),
                        "tokens": tokens_int,
                    }
                    fout.write(json.dumps(rec) + "\n")
                    sample_id += 1
    
    print(f"Done. Saved {split_name} JSONL: {out_path}")

# =========================
# train/val/test splits
# =========================
ns = SimpleNamespace(dataset=TARGET_DATASET, data_root=UCR_ROOT, root_path=UCR_ROOT,
                     num_workers=0, seq_len=None, seed=42)
train_ls, val_ls, test_ls, param = ucr_data_provider(ns)
train_loader = train_ls[0]
val_loader = val_ls[0]
test_loader = test_ls[0]

#  Nasdaq data 
# nasdaq_path = os.path.join(DATA_DIR, "Nasdaq")
# args = SimpleNamespace(ucr_root=nasdaq_path, batch_size=32)
# train_loader, val_loader, test_loader, _ = nasdaq_data_provider(args.ucr_root, args.batch_size)

# etth1_path = os.path.join(DATA_DIR, "ETTh1")
# args = SimpleNamespace(ucr_root=etth1_path, batch_size=32)
# train_loader, val_loader, test_loader, _ = etth1_data_provider(args.ucr_root, args.batch_size)

# =========================
# Select ckpt
# =========================
best_list = sorted(glob.glob(os.path.join(CKPT_DIR, "TimeTok_best_val_epoch*.pt")))
if len(best_list) == 0:
    best_list = sorted(glob.glob(os.path.join(CKPT_DIR, "TimeTok_best_val_epoch*.pt")))
assert len(best_list) > 0, f"No best ckpt: {CKPT_DIR}"
BEST_CKPT = max(best_list, key=os.path.getmtime)
# print the file name of the BEST_CKPT
print(f"Using checkpoint: {os.path.basename(BEST_CKPT)}")

CFG_PATH  = os.path.join(CKPT_DIR, "config_snapshot.json")
assert os.path.isfile(CFG_PATH), f"config_snapshot.json: {CFG_PATH}"

print("BEST_CKPT:", BEST_CKPT)

with open(CFG_PATH, "r") as f:
    cfg = json.load(f)

cfg["encoder"]["module_dict"]["enc_patch_emb"]["patch_sizes"] = [Pz]
cfg["decoder"]["module_dict"]["dec_noise_patch_emb"]["patch_sizes"] = [Pz]
cfg["decoder"]["module_dict"]["dec_to_patches"]["patch_sizes"] = [Pz]

tsgen: TimeTok = instantiate(cfg)
state = torch.load(BEST_CKPT, map_location=DEVICE)
tsgen.load_state_dict(state)
tsgen = tsgen.to(DEVICE).eval()


os.makedirs(OUT_DIR, exist_ok=True)

splits_info = [
    ("train", train_loader, f"{TARGET_DATASET}_pz{Pz}_train_fsq_tokens.jsonl"),
    ("val", val_loader, f"{TARGET_DATASET}_pz{Pz}_val_fsq_tokens.jsonl"),
    ("test", test_loader, f"{TARGET_DATASET}_pz{Pz}_test_fsq_tokens.jsonl")
]

# =========================
# =========================
for split_name, data_loader, filename in splits_info:
    out_path = os.path.join(OUT_DIR, filename)
    
    meta = {
        "dataset": TARGET_DATASET,
        "tokenizer_dataset": TOKENIZER_DATASET,
        "split": split_name,
        "ckpt": BEST_CKPT,
        "cfg_path": CFG_PATH,
        "patch_size": Pz,
    }
    
    meta_path = out_path + ".meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    
    tokenize_split(
        data_loader=data_loader,
        split_name=split_name,
        tsgen=tsgen,
        out_path=out_path,
        dataset_name=TARGET_DATASET,
        patch_size=Pz,
        enable_bf16=ENABLE_BF16,
        dataset_id=dataset_id,
        return_eval_keep_k=return_eval_keep_k
    )

print(f"All splits completed! Files saved in: {OUT_DIR}")
print(f"Generated files:")
for split_name, _, filename in splits_info:
    print(f"  - {filename}")
    print(f"  - {filename}.meta.json")

# Copy the CKPT and CFG to the OUT_DIR
import os
import shutil

shutil.copy(BEST_CKPT, OUT_DIR)
shutil.copy(CFG_PATH, OUT_DIR)
print(f"Copied {BEST_CKPT} and {CFG_PATH} to {OUT_DIR}")